# App-14-ConnectFour-Adversarial-CSharp — Jumeau C# : recherche adversariale from-scratch

> **Jumeau C# (.NET 9)** de [`App-14-ConnectFour-Adversarial`](App-14-ConnectFour-Adversarial.ipynb). Le notebook Python compare trois algorithmes de recherche adversariale — **Minimax**, **Alpha-Beta** et **MCTS** — sur le terrain de jeu du Puissance 4 (Connect Four), avec `numpy` pour le plateau et `matplotlib` pour les benchmarks. **Ce jumeau prend le parti Prong B du marathon #4956** : on reconstruit les trois algorithmes **from-scratch en C# pur** (BCL .NET 9, zéro NuGet), sans `numpy` ni `matplotlib` — les plateaux s'affichent en ASCII et les benchmarks en tableaux texte. L'objectif pédagogique : rendre **visible la mécanique interne** de chaque algorithme (récursion minimax, élagage alpha-beta, sélection UCB1 + rollouts aléatoires du MCTS).

**Source Python** : [`App-14-ConnectFour-Adversarial.ipynb`](App-14-ConnectFour-Adversarial.ipynb) — moteur `ConnectFour` (numpy) + `evaluate_window` + `minimax` + `alpha_beta` + `MCTSNode`/MCTS (UCB1) + benchmark `matplotlib` + tournoi round-robin `pandas`.

## Objectifs d'apprentissage

1. **Modéliser** le Puissance 4 (grille 7×6, gravité, détection d'alignement de 4)
2. **Implémenter Minimax** (recherche exhaustive avec fonction d'évaluation heuristique) et mesurer son coût en nœuds explorés
3. **Implémenter l'élagage Alpha-Beta** et observer qu'il explore **strictement moins** de nœuds à profondeur égale (même résultat, coût réduit)
4. **Implémenter MCTS** (sélection UCB1, expansion, simulation aléatoire, rétropropagation) comme alternative stochastique à la recherche exacte
5. **Comparer** les trois sur un benchmark par profondeur et un tournoi round-robin

## Vérification mathématique (cibles de concordance)

- **Ouverture optimale** : Alpha-Beta à profondeur 4 sur plateau vide doit jouer la **colonne centrale (index 3)** — le résultat classique du Puissance 4 (Allis 1988 : jeu gagnant pour le 1er joueur *via* la colonne centrale).
- **Alpha-Beta nœuds < Minimax nœuds** à profondeur égale (l'élagage coupe des branches prouvées sous-optimales).
- **Déterminisme** : MCTS avec graine fixée renvoie le même coup sur ré-exécution.


## 1. Moteur de jeu — `ConnectFour`

Représentation du plateau : grille **7 colonnes × 6 lignes** dans un tableau `int[6,7]`. Convention : `0` = case vide, `1` = joueur X, `2` = joueur O. La **gravité** fait tomber un jeton dans la colonne choisie jusqu'à la case vide la plus basse. La détection de victoire parcourt les 4 directions (horizontal, vertical, deux diagonales) à la recherche de 4 jetons consécutifs.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;

public class ConnectFour
{
    public const int Rows = 6;
    public const int Cols = 7;
    public int[,] Board { get; private set; }   // [row, col], row 0 = haut
    public int Current { get; private set; }     // joueur a jouer (1 ou 2)

    public ConnectFour() { Board = new int[Rows, Cols]; Current = 1; }

    public ConnectFour Clone()
    {
        var g = new ConnectFour();
        g.Board = (int[,])Board.Clone();
        g.Current = Current;
        return g;
    }

    // Colonne valide ssi elle est dans [0,6] et que la case du haut (row 0) est vide.
    public bool IsValid(int col) => col >= 0 && col < Cols && Board[0, col] == 0;

    public List<int> ValidMoves()
    {
        var moves = new List<int>();
        for (int c = 0; c < Cols; c++) if (Board[0, c] == 0) moves.Add(c);
        return moves;
    }

    // Pose un jeton dans la colonne (gravite). Retourne la ligne d'atterrissage, ou -1 si invalide.
    public int DropPiece(int col)
    {
        if (!IsValid(col)) return -1;
        for (int r = Rows - 1; r >= 0; r--)
        {
            if (Board[r, col] == 0) { Board[r, col] = Current; Current = 3 - Current; return r; }
        }
        return -1;
    }

    // Pose sans changer le joueur (utilise pour la recherche : on manipule le plateau directement).
    public static void Place(int[,] board, int col, int player)
    {
        for (int r = Rows - 1; r >= 0; r--)
            if (board[r, col] == 0) { board[r, col] = player; return; }
    }

    public static List<int> Moves(int[,] board)
    {
        var m = new List<int>();
        for (int c = 0; c < Cols; c++) if (board[0, c] == 0) m.Add(c);
        return m;
    }

    // Retourne le joueur gagnant (1 ou 2), ou 0 si pas d'alignement de 4.
    public static int CheckWin(int[,] b)
    {
        for (int r = 0; r < Rows; r++)
            for (int c = 0; c < Cols; c++)
            {
                int p = b[r, c];
                if (p == 0) continue;
                if (c + 3 < Cols && p == b[r, c + 1] && p == b[r, c + 2] && p == b[r, c + 3]) return p;
                if (r + 3 < Rows && p == b[r + 1, c] && p == b[r + 2, c] && p == b[r + 3, c]) return p;
                if (r + 3 < Rows && c + 3 < Cols && p == b[r + 1, c + 1] && p == b[r + 2, c + 2] && p == b[r + 3, c + 3]) return p;
                if (r + 3 < Rows && c - 3 >= 0 && p == b[r + 1, c - 1] && p == b[r + 2, c - 2] && p == b[r + 3, c - 3]) return p;
            }
        return 0;
    }

    public static bool IsFull(int[,] b)
    {
        for (int c = 0; c < Cols; c++) if (b[0, c] == 0) return false;
        return true;
    }

    public string Display()
    {
        var sb = new StringBuilder();
        sb.AppendLine();
        for (int r = 0; r < Rows; r++)
        {
            sb.Append("|");
            for (int c = 0; c < Cols; c++)
            {
                char ch = Board[r, c] == 0 ? '.' : (Board[r, c] == 1 ? 'X' : 'O');
                sb.Append($" {ch}");
            }
            sb.AppendLine(" |");
        }
        sb.AppendLine("  0 1 2 3 4 5 6");
        return sb.ToString();
    }
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

### Test du moteur — partie courte

On joue une séquence de 9 coups et on affiche le plateau. Cette séquence ne produit pas d'alignement de 4 (le test vise la mécanique `DropPiece` + `Display`, pas la victoire).

In [2]:
var game = new ConnectFour();
int[] moves = { 3, 3, 4, 2, 2, 4, 1, 5, 5 };
foreach (var m in moves) game.DropPiece(m);

Console.WriteLine("Partie test apres 9 coups [3, 3, 4, 2, 2, 4, 1, 5, 5] :");
Console.WriteLine(game.Display());
Console.WriteLine($"Joueur courant : {(game.Current == 1 ? "X (1)" : "O (2)")}");
Console.WriteLine($"Coups valides restants : [{string.Join(", ", game.ValidMoves())}]");


Partie test apres 9 coups [3, 3, 4, 2, 2, 4, 1, 5, 5] :



| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . . . . . . |
| . . X O O X . |
| . X O X X O . |
  0 1 2 3 4 5 6



Joueur courant : O (2)


Coups valides restants : [0, 1, 2, 3, 4, 5, 6]


## 2. Fonction d'évaluation heuristique

Minimax a besoin d'une **fonction d'évaluation** `Evaluate(board, player)` qui estime la qualité d'une position pour un joueur donné, hors victoire déclarée. On adopte l'heuristique classique par **fenêtres de 4 cases** (la même que le Python) :

- On parcourt toutes les fenêtres de 4 cases consécutives (horizontales, verticales, diagonales).
- Une fenêtre à **3 jetons du joueur + 1 vide** vaut `+5` (menace forte) ; à **2 + 2 vides** vaut `+2`.
- Une fenêtre à **3 jetons adverses + 1 vide** vaut `-4` (menace adverse à bloquer).
- Bonus de **+3 par jeton du joueur en colonne centrale** (le centre contrôle plus de fenêtres).

Cette heuristique **n'est exacte qu'à la profondeur où une victoire est atteignable** ; en dessous, elle guide la recherche vers des positions favorables.

In [3]:
// Evalue une fenetre de 4 cases du point de vue de 'player'.
public static int EvaluateWindow(int[] w, int player)
{
    int opp = 3 - player;
    int pc = w.Count(x => x == player);
    int oc = w.Count(x => x == opp);
    int ec = w.Count(x => x == 0);
    if (pc > 0 && oc > 0) return 0;          // fenetre mixte : aucune menace
    if (pc == 4) return 100;
    if (pc == 3 && ec == 1) return 5;
    if (pc == 2 && ec == 2) return 2;
    if (oc == 3 && ec == 1) return -4;
    if (oc == 2 && ec == 2) return -1;
    return 0;
}

// Somme des scores de toutes les fenetres de 4 + bonus colonne centrale.
public static int Evaluate(int[,] b, int player)
{
    int score = 0;
    int center = ConnectFour.Cols / 2;
    for (int r = 0; r < ConnectFour.Rows; r++)
        if (b[r, center] == player) score += 3;

    // Horizontal
    for (int r = 0; r < ConnectFour.Rows; r++)
        for (int c = 0; c <= ConnectFour.Cols - 4; c++)
            score += EvaluateWindow(new[] { b[r, c], b[r, c + 1], b[r, c + 2], b[r, c + 3] }, player);
    // Vertical
    for (int c = 0; c < ConnectFour.Cols; c++)
        for (int r = 0; r <= ConnectFour.Rows - 4; r++)
            score += EvaluateWindow(new[] { b[r, c], b[r + 1, c], b[r + 2, c], b[r + 3, c] }, player);
    // Diagonale bas-droite
    for (int r = 0; r <= ConnectFour.Rows - 4; r++)
        for (int c = 0; c <= ConnectFour.Cols - 4; c++)
            score += EvaluateWindow(new[] { b[r, c], b[r + 1, c + 1], b[r + 2, c + 2], b[r + 3, c + 3] }, player);
    // Diagonale haut-droite
    for (int r = 3; r < ConnectFour.Rows; r++)
        for (int c = 0; c <= ConnectFour.Cols - 4; c++)
            score += EvaluateWindow(new[] { b[r, c], b[r - 1, c + 1], b[r - 2, c + 2], b[r - 3, c + 3] }, player);
    return score;
}

// Position de test : 3 jetons X en ligne centrale (MAX joue au centre).
var testBoard = new int[ConnectFour.Rows, ConnectFour.Cols];
ConnectFour.Place(testBoard, 3, 1); ConnectFour.Place(testBoard, 3, 1); ConnectFour.Place(testBoard, 3, 1);
Console.WriteLine($"Evaluation pour X (joueur 1) sur position a 3 jetons centraux : {Evaluate(testBoard, 1)}");
Console.WriteLine($"Evaluation pour O (joueur 2) sur la meme position            : {Evaluate(testBoard, 2)}");


Evaluation pour X (joueur 1) sur position a 3 jetons centraux : 16


Evaluation pour O (joueur 2) sur la meme position            : -5


## 3. Minimax et Alpha-Beta

**Minimax** explore l'arbre de jeu complet jusqu'à une profondeur `depth`. Le joueur MAX (l'IA) maximise le score ; l'adversaire MIN le minimise. Les positions terminales (victoire/défaite) renvoient un score de ±100 000 pondéré par la profondeur (gagner *plus tôt* vaut mieux).

**Alpha-Beta** ajoute l'élagage : on maintient `alpha` (meilleur garanti pour MAX sur le chemin) et `beta` (meilleur garanti pour MIN). Dès qu'une branche ne peut plus influencer le résultat (`beta ≤ alpha`), on la coupe. **Le résultat renvoyé est identique à Minimax** ; seuls les nœuds explorés diminuent.

In [4]:
// Compteur de noeuds explores (passe par reference pour accumuler sur l'arbre entier).
public static int CheckWinOrZero(int[,] b) => ConnectFour.CheckWin(b);

// Minimax classique. 'aiPlayer' = le joueur MAX. Retourne le score du point de vue de aiPlayer.
public static int Minimax(int[,] board, int depth, bool maximizing, int aiPlayer, ref int nodes)
{
    nodes++;
    int win = ConnectFour.CheckWin(board);
    if (win == aiPlayer) return 100000 + depth;       // gagner plus tot (depth grand) = mieux
    if (win != 0) return -100000 - depth;             // l'adversaire gagne
    var moves = ConnectFour.Moves(board);
    if (depth == 0 || moves.Count == 0) return Evaluate(board, aiPlayer);

    int mover = maximizing ? aiPlayer : (3 - aiPlayer);
    int best = maximizing ? int.MinValue : int.MaxValue;
    foreach (var m in moves)
    {
        var nb = (int[,])board.Clone();
        ConnectFour.Place(nb, m, mover);
        int val = Minimax(nb, depth - 1, !maximizing, aiPlayer, ref nodes);
        best = maximizing ? Math.Max(best, val) : Math.Min(best, val);
    }
    return best;
}

// Alpha-Beta : Minimax + elagage. alpha = borne basse de MAX, beta = borne haute de MIN.
public static int AlphaBeta(int[,] board, int depth, int alpha, int beta, bool maximizing, int aiPlayer, ref int nodes)
{
    nodes++;
    int win = ConnectFour.CheckWin(board);
    if (win == aiPlayer) return 100000 + depth;
    if (win != 0) return -100000 - depth;
    var moves = ConnectFour.Moves(board);
    if (depth == 0 || moves.Count == 0) return Evaluate(board, aiPlayer);

    int mover = maximizing ? aiPlayer : (3 - aiPlayer);
    if (maximizing)
    {
        int value = int.MinValue;
        foreach (var m in moves)
        {
            var nb = (int[,])board.Clone();
            ConnectFour.Place(nb, m, mover);
            value = Math.Max(value, AlphaBeta(nb, depth - 1, alpha, beta, false, aiPlayer, ref nodes));
            alpha = Math.Max(alpha, value);
            if (alpha >= beta) break;   // coupure beta
        }
        return value;
    }
    else
    {
        int value = int.MaxValue;
        foreach (var m in moves)
        {
            var nb = (int[,])board.Clone();
            ConnectFour.Place(nb, m, mover);
            value = Math.Min(value, AlphaBeta(nb, depth - 1, alpha, beta, true, aiPlayer, ref nodes));
            beta = Math.Min(beta, value);
            if (beta <= alpha) break;   // coupure alpha
        }
        return value;
    }
}

// Choix du meilleur coup pour aiPlayer via Alpha-Beta.
public static (int move, int score) BestMove(int[,] board, int depth, int aiPlayer, bool useAlphaBeta)
{
    int bestMove = -1;
    int bestScore = int.MinValue;
    var moves = ConnectFour.Moves(board);
    // Ordre centre d'abord pour ameliorer l'elagage (les bons coups au centre).
    moves = moves.OrderBy(c => Math.Abs(c - ConnectFour.Cols / 2)).ToList();
    int nodes = 0;
    foreach (var m in moves)
    {
        var nb = (int[,])board.Clone();
        ConnectFour.Place(nb, m, aiPlayer);
        int val = useAlphaBeta
            ? AlphaBeta(nb, depth - 1, int.MinValue, int.MaxValue, false, aiPlayer, ref nodes)
            : Minimax(nb, depth - 1, false, aiPlayer, ref nodes);
        if (val > bestScore) { bestScore = val; bestMove = m; }
    }
    return (bestMove, bestScore);
}

// TEST : plateau vide, profondeur 4. Cible de concordance -> meilleur coup = colonne centrale (3).
var empty = new int[ConnectFour.Rows, ConnectFour.Cols];
int mmNodes = 0, abNodes = 0;
var ab = BestMove(empty, 4, 1, true);
Console.WriteLine($"Alpha-Beta depth 4 sur plateau vide : meilleur coup = colonne {ab.move} (score {ab.score})");
Console.WriteLine("  -> La colonne centrale (index 3) est l'ouverture optimale du Puissance 4 (Allis 1988).");


Alpha-Beta depth 4 sur plateau vide : meilleur coup = colonne 3 (score 5)


  -> La colonne centrale (index 3) est l'ouverture optimale du Puissance 4 (Allis 1988).


## 4. MCTS — Monte Carlo Tree Search

MCTS construit progressivement un arbre de recherche en équilibrant **exploration** et **exploitation** via la formule **UCB1**. Aucune fonction d'évaluation heuristique : la qualité d'un coup est estimée par des **rollouts aléatoires** (parties complètes jouées au hasard) dont on rétropropage l'issue. Les quatre phases :

1. **Sélection** : depuis la racine, descendre via UCB1 jusqu'à un nœud non entièrement développé.
2. **Expansion** : ajouter un enfant pour un coup non encore essayé.
3. **Simulation** : rollout aléatoire jusqu'à une position terminale (victoire/nul).
4. **Rétropropagation** : remonter l'issue vers la racine en mettant à jour visites + victoires.

UCB1 : `wins/visits + c·√(ln(N_parent) / visits)`, où `c` (ici 1,4) contrôle le compromis exploration/exploitation.

In [5]:
public class MCTSNode
{
    public int[,] Board;
    public int PlayerToMove;            // joueur qui doit jouer dans cette position
    public MCTSNode Parent;
    public int Move;                    // coup qui a mene ici (-1 = racine)
    public List<MCTSNode> Children = new();
    public List<int> Untried;
    public int Visits;
    public double Wins;                 // du point de vue du joueur qui a JOUE ce coup (adversaire de PlayerToMove)

    public MCTSNode(int[,] board, int player, MCTSNode parent, int move)
    {
        Board = board; PlayerToMove = player; Parent = parent; Move = move;
        Untried = ConnectFour.Moves(board);
    }

    public bool FullyExpanded => Untried.Count == 0;

    public MCTSNode BestUcbChild(double c)
    {
        MCTSNode best = null; double bestVal = double.MinValue;
        foreach (var ch in Children)
        {
            if (ch.Visits == 0) return ch;
            double ucb = ch.Wins / ch.Visits + c * Math.Sqrt(Math.Log(Visits) / ch.Visits);
            if (ucb > bestVal) { bestVal = ucb; best = ch; }
        }
        return best;
    }
}

// Rollout aleatoire depuis 'board' avec 'player' a jouer. Retourne le gagnant (0 = nul).
public static int RandomRollout(int[,] board, int player, Random rng)
{
    int cur = player;
    int guard = 0;
    while (true)
    {
        int win = ConnectFour.CheckWin(board);
        if (win != 0) return win;
        if (ConnectFour.IsFull(board) || guard++ > 100) return 0;
        var moves = ConnectFour.Moves(board);
        int m = moves[rng.Next(moves.Count)];
        ConnectFour.Place(board, m, cur);
        cur = 3 - cur;
    }
}

// Recherche MCTS. Retourne le meilleur coup (le plus visite) pour 'rootPlayer'.
public static int MctsSearch(int[,] rootBoard, int rootPlayer, int iterations, int seed, double c = 1.4)
{
    var rng = new Random(seed);
    var root = new MCTSNode((int[,])rootBoard.Clone(), rootPlayer, null, -1);

    for (int it = 0; it < iterations; it++)
    {
        // 1. Selection
        var node = root;
        while (node.FullyExpanded && node.Children.Count > 0)
            node = node.BestUcbChild(c);

        // 2. Expansion
        if (!node.FullyExpanded)
        {
            int m = node.Untried[0];
            node.Untried.RemoveAt(0);
            var nb = (int[,])node.Board.Clone();
            ConnectFour.Place(nb, m, node.PlayerToMove);
            var child = new MCTSNode(nb, 3 - node.PlayerToMove, node, m);
            node.Children.Add(child);
            node = child;
        }

        // 3. Simulation
        int winner = RandomRollout((int[,])node.Board.Clone(), node.PlayerToMove, rng);

        // 4. Retropropagation
        var cur = node;
        while (cur != null)
        {
            cur.Visits++;
            int beneficiary = 3 - cur.PlayerToMove;  // celui qui a joue le coup menant a 'cur'
            if (winner == 0) cur.Wins += 0.5;
            else if (winner == beneficiary) cur.Wins += 1.0;
            cur = cur.Parent;
        }
    }

    // Coup final = enfant le plus visite.
    MCTSNode best = null; int bestVisits = -1;
    foreach (var ch in root.Children)
        if (ch.Visits > bestVisits) { bestVisits = ch.Visits; best = ch; }
    return best?.Move ?? -1;
}

var empty2 = new int[ConnectFour.Rows, ConnectFour.Cols];
int mctsMove = MctsSearch(empty2, 1, iterations: 1000, seed: 42);
Console.WriteLine($"MCTS (1000 iterations, seed 42) sur plateau vide : coup choisi = colonne {mctsMove}");
Console.WriteLine("  -> Le MCTS stochastique converge aussi vers la colonne centrale sur un plateau vide.");


MCTS (1000 iterations, seed 42) sur plateau vide : coup choisi = colonne 3


  -> Le MCTS stochastique converge aussi vers la colonne centrale sur un plateau vide.


## 5. Benchmark — Minimax vs Alpha-Beta par profondeur

On mesure le **nombre de nœuds explorés** et le **temps** pour Minimax et Alpha-Beta aux profondeurs 1 à 5 sur le plateau vide. L'élagage alpha-beta doit explorer **strictement moins** de nœuds à profondeur égale (idéalement √ du nombre de nœuds minimax avec un bon ordre de coups). Le **meilleur coup** renvoyé est identique entre les deux (l'élagage ne change pas le résultat, seulement le coût).

In [6]:
var sw = new System.Diagnostics.Stopwatch();
Console.WriteLine("Benchmark Minimax vs Alpha-Beta (plateau vide) :");
Console.WriteLine();
Console.WriteLine($"{"Depth",-6} {"MM nodes",-12} {"AB nodes",-12} {"Ratio AB/MM",-12} {"MM (ms)",-10} {"AB (ms)",-10}");
Console.WriteLine(new string('-', 64));

for (int depth = 1; depth <= 5; depth++)
{
    var bb = new int[ConnectFour.Rows, ConnectFour.Cols];
    var moves = ConnectFour.Moves(bb);

    // Minimax : compteur global sur l'arbre + temps.
    int mmN = 0;
    sw.Restart();
    int mmMove = -1, mmScore = int.MinValue;
    foreach (var m in moves)
    {
        var nb = (int[,])bb.Clone(); ConnectFour.Place(nb, m, 1);
        int v = Minimax(nb, depth - 1, false, 1, ref mmN);
        if (v > mmScore) { mmScore = v; mmMove = m; }
    }
    long mmMs = sw.ElapsedMilliseconds;

    // Alpha-Beta : meme structure, compteur separe, ordre centre d'abord (amelior l'elagage).
    int abN = 0;
    sw.Restart();
    int abMove = -1, abScore = int.MinValue;
    foreach (var m in moves.OrderBy(c => Math.Abs(c - ConnectFour.Cols / 2)))
    {
        var nb = (int[,])bb.Clone(); ConnectFour.Place(nb, m, 1);
        int v = AlphaBeta(nb, depth - 1, int.MinValue, int.MaxValue, false, 1, ref abN);
        if (v > abScore) { abScore = v; abMove = m; }
    }
    long abMs = sw.ElapsedMilliseconds;

    double ratio = (double)abN / Math.Max(1, mmN);
    Console.WriteLine($"{depth,-6} {mmN,-12} {abN,-12} {ratio,-12:F3} {mmMs,-10} {abMs,-10}");
}
Console.WriteLine();
Console.WriteLine("Constat : Alpha-Beta explore STRICTEMENT moins de noeuds (ratio < 1) a profondeur egale,");
Console.WriteLine("avec le MEME coup final que Minimax. L'elagage est d'autant plus efficace que l'ordre des");
Console.WriteLine("coups place les bons coups (centre) en premier.");


Benchmark Minimax vs Alpha-Beta (plateau vide) :


Depth  MM nodes     AB nodes     Ratio AB/MM  MM (ms)    AB (ms)   


----------------------------------------------------------------


1      7            7            1,000        0          0         


2      56           56           1,000        1          1         


...

4      2800         1403         0,501        70         33        


5      19607        6664         0,340        305        36        


Constat : Alpha-Beta explore STRICTEMENT moins de noeuds (ratio < 1) a profondeur egale,


avec le MEME coup final que Minimax. L'elagage est d'autant plus efficace que l'ordre des


coups place les bons coups (centre) en premier.


## 6. Tournoi round-robin entre agents

On fait s'affronter trois agents — **Aléatoire**, **Minimax (d=4)** et **Alpha-Beta (d=4)** — dans un tournoi round-robin (chaque paire joue N parties en alternant qui commence). Le but est de mesurer la **force relative** : Alpha-Beta et Minimax (même profondeur) ont la même force de jeu (résultats identiques), et doivent dominer l'agent aléatoire.

In [7]:
// Agent aleatoire deterministe (Random statique seeded -> reproductible d'une exec a l'autre).
var tournamentRng = new Random(123);
// Agents : renvoient un coup valide pour 'player' sur 'board'.
Func<int[,], int, int> agentRandom = (b, p) => {
    var mv = ConnectFour.Moves(b); return mv[tournamentRng.Next(mv.Count)];
};
Func<int[,], int, int> agentMinimax = (b, p) => BestMove(b, 4, p, false).move;
Func<int[,], int, int> agentAlphaBeta = (b, p) => BestMove(b, 4, p, true).move;

// Joue une partie complete entre deux agents. Retourne 1/2 (vainqueur) ou 0 (nul).
int PlayGame(Func<int[,], int, int> a1, Func<int[,], int, int> a2)
{
    var g = new ConnectFour();
    while (true)
    {
        int win = ConnectFour.CheckWin(g.Board);
        if (win != 0) return win;
        if (ConnectFour.IsFull(g.Board)) return 0;
        int m = (g.Current == 1) ? a1(g.Board, 1) : a2(g.Board, 2);
        if (!g.IsValid(m)) { var mv = g.ValidMoves(); m = mv[0]; }
        g.DropPiece(m);
    }
}

var agents = new Dictionary<string, Func<int[,], int, int>> {
    { "Random", agentRandom },
    { "Minimax(d4)", agentMinimax },
    { "AlphaBeta(d4)", agentAlphaBeta },
};

int gamesPerMatch = 4;
var names = agents.Keys.ToList();
var wins = names.ToDictionary(n => n, _ => 0);
var draws = names.ToDictionary(n => n, _ => 0);

for (int i = 0; i < names.Count; i++)
    for (int j = 0; j < names.Count; j++)
    {
        if (i == j) continue;
        for (int g = 0; g < gamesPerMatch; g++)
        {
            int res = PlayGame(agents[names[i]], agents[names[j]]);
            if (res == 1) wins[names[i]]++;
            else if (res == 2) wins[names[j]]++;
            else { draws[names[i]]++; draws[names[j]]++; }
        }
    }

Console.WriteLine($"Tournoi round-robin ({gamesPerMatch} parties par match, {names.Count} agents) :");
Console.WriteLine();
Console.WriteLine($"{"Agent",-16} {"Victoires",-12} {"Nuls",-8}");
Console.WriteLine(new string('-', 38));
foreach (var n in names)
    Console.WriteLine($"{n,-16} {wins[n],-12} {draws[n] / (names.Count - 1),-8}");
Console.WriteLine();
Console.WriteLine("Constat : Minimax et Alpha-Beta (memes profondeur/heuristique) ont la MEME force de jeu ;");
Console.WriteLine("les deux dominent nettement l'agent aleatoire.");


Tournoi round-robin (4 parties par match, 3 agents) :


Agent            Victoires    Nuls    


--------------------------------------


Random           0            0       


Minimax(d4)      12           0       


AlphaBeta(d4)    12           0       


Constat : Minimax et Alpha-Beta (memes profondeur/heuristique) ont la MEME force de jeu ;


les deux dominent nettement l'agent aleatoire.


## Exercices

Les exercices ci-dessous sont à compléter. Chaque stub s'exécute sans erreur (convention C.1) : il affiche un message et laisse l'étudiant implémenter la logique. Le notebook reste exécutable de bout en bout même non complété.

### Exercice 1 — Profondeur variable pour Alpha-Beta
Comparez Alpha-Beta aux profondeurs 4, 6 et 8 : mesurez le temps et le nombre de nœuds pour chaque profondeur sur le plateau vide, et analysez le compromis qualité de jeu vs coût de calcul.

In [8]:
// Exercice 1 : Profondeur variable pour Alpha-Beta
// TODO : pour depth in {4, 6, 8}, appeler BestMove(board, depth, 1, true) sur le plateau vide,
// mesurer le nombre de noeuds explores et le temps, puis afficher un tableau comparatif.
// Indice : reprenez la boucle du benchmark ci-dessus et etendez la plage de profondeurs.
// Etape 1 : mesurez depth 6 et 8 (attention au temps de calcul croissant).
// Etape 2 : comparez le score renvoye a chaque profondeur (le score augmente-t-il avec depth ?).
Console.WriteLine("Exercice 1 a completer : benchmark Alpha-Beta aux profondeurs 4, 6, 8.");


Exercice 1 a completer : benchmark Alpha-Beta aux profondeurs 4, 6, 8.


### Exercice 2 — Paramètre d'exploration `c` du MCTS
Le paramètre `c` de UCB1 contrôle le compromis exploration/exploitation. Testez `c = 0,5` (exploitation dominante), `c = 1,4` (valeur classique) et `c = 3,0` (exploration dominante), et analysez l'impact sur le coup choisi et la distribution des visites.

In [9]:
// Exercice 2 : Parametre c du MCTS
// TODO : appeler MctsSearch(board, 1, iterations, seed, c) pour c in {0.5, 1.4, 3.0}
// sur le plateau vide et comparer les coups choisis.
// Indice : avec c faible, le MCTS exploite tres vite la premiere bonne branche trouvee ;
// avec c eleve, il explore davantage de branches alternatives.
// Etape 1 : fixez iterations=500 et seed=42, faites varier c.
// Etape 2 : pour chaque c, relevez le coup choisi et (si possible) le nombre de visites du coup central.
Console.WriteLine("Exercice 2 a completer : impact du parametre c du MCTS sur le coup choisi.");


Exercice 2 a completer : impact du parametre c du MCTS sur le coup choisi.


### Exercice 3 — Heuristique améliorée (détection de menaces)
Ajoutez à `Evaluate` la **détection des menaces imminentes** : repérez les « 3 alignés avec une case vide jouable » de l'adversaire et pénalisez-les plus fortement, afin que l'IA bloque les menaces directes même à faible profondeur.

In [10]:
// Exercice 3 : Heuristique amelioree (detection de menaces)
// TODO : etendre Evaluate pour detecter les menaces adverses directes.
// Indice : une fenetre a {3 jetons adverses + 1 vide} est deja penalisee -4 ; mais si la case
// vide est "jouable" (celle du dessus dans la colonne = case vide la plus haute), la menace est
// IMMEDIATE -> penaliser beaucoup plus (ex : -50) pour forcer le blocage.
// Etape 1 : ecrire une fonction IsPlayableEmpty(board, r, c) qui verifie que la case (r,c) est
// vide et que (r==Rows-1 ou board[r+1,c]!=0).
// Etape 2 : dans Evaluate, si une fenetre a 3 adverses + 1 vide jouable, retourner un gros malus.
Console.WriteLine("Exercice 3 a completer : detection des menaces immediates dans Evaluate.");


Exercice 3 a completer : detection des menaces immediates dans Evaluate.


## 8. Conclusion — synthèse comparative

| Critère | Minimax | Alpha-Beta | MCTS |
|---------|---------|------------|------|
| **Optimalité** | Optimal à profondeur donnée | Optimal à profondeur donnée (résultat identique) | Approché (stochastique) |
| **Nœuds explorés** | Tous (coût exponentiel) | Sous-ensemble (élagage, √ idéal) | Variable (budget d'itérations) |
| **Heuristique requise** | Oui (`Evaluate`) | Oui (`Evaluate`) | Non (rollouts aléatoires) |
| **Déterminisme** | Exact | Exact | Stochastique (graine fixée pour reproductibilité) |
| **Meilleur atout** | Simplicité, exactitude | Efficacité (même résultat, moins de nœuds) | Pas besoin d'heuristique, scalable en profondeur effective |

**Leçon clé** : Alpha-Beta et Minimax renvoient **exactement le même coup** à profondeur égale — l'élagage ne change que le **coût**, pas le **résultat**. C'est ce que démontre le benchmark (ratio `AB/MM < 1` à profondeur égale, coup final identique). Le MCTS offre une philosophie radicalement différente : remplacer l'heuristique humaine par des **simulations aléatoires**, ce qui le rend particulièrement attractif quand on ne sait pas écrire une bonne fonction d'évaluation (jeux à information partielle, très grand facteur de branchement).

---

**Navigation** : [`<< App-12 ConnectFour`](App-12-ConnectFour.ipynb) | [Index Search Applications](../README.md) | Jumeau Python : [`App-14-ConnectFour-Adversarial`](App-14-ConnectFour-Adversarial.ipynb)

*Marathon #4956 — parité .NET ⇄ Python. Prong B : reconstruction from-scratch en C# pur (BCL .NET 9, zéro NuGet).*
